### Notes

In [1]:

import os
import sys
sys.path.append('/home/hengjie/DL_projects/brainmets/1210_wip/0107')
from myRTDoseReader import myRTDoseReader

dir_base = "/database/brainmets/dicom/SBRT_research_mim_export_20251209_organized"


# # 0944.1998.11 (fail)
# path_dict = {
#     "MR":     "SRS0944/1998-11__Studies/SRS0944_SRS0944_MR_1998-11-12_000000_._axial_n45__00000",
#     "RTDOSE": "SRS0944/1998-11__Studies/SRS0944_SRS0944_RTDOSE_1998-11-12_000000_._._n1__00000",
# }

# 3126.2021.01 (success)
# path_dict = {
#     "MR":     "SRS3126/2011-01__Studies/SRS3126_SRS3126_MR_2011-01-07_083714_MR.STEREO.BRAIN_AX.T1.PG_n51__00000",
#     "RTDOSE": "SRS3126/2011-01__Studies/SRS3126_SRS3126_RTDOSE_2011-01-07_083714_MR.STEREO.BRAIN_._n1__00000",
# }


# 1669.2002.03 (fail)
path_dict = {
    "MR":     "SRS1669/2002-03__Studies/SRS1669_SRS1669_MR_2002-03-12_000000_._axial1_n22__00000",
    "RTDOSE": "SRS1669/2002-03__Studies/SRS1669_SRS1669_RTDOSE_2002-03-12_000000_._._n1__00000",
}

# 1669.2010.10 (success)
# path_dict = {
#     "MR":     "SRS1669/2010-10__Studies/SRS1669_SRS1669_MR_2010-10-04_091759_MR.STEREO.BRAIN_Ax.T1.PG_n51__00000",
#     "RTDOSE": "SRS1669/2010-10__Studies/SRS1669_SRS1669_RTDOSE_2010-10-04_091759_MR.STEREO.BRAIN_._n1__00000",
# }

folder_mr = os.path.join(dir_base, path_dict["MR"])
folder_rtdose = os.path.join(dir_base, path_dict["RTDOSE"])

In [4]:
import os
import SimpleITK as sitk


def _get_dicom_tag_as_str(path: str, tag: str, load_private: bool = True) -> str | None:
    """Read a single DICOM tag as string via SimpleITK without loading pixels."""
    r = sitk.ImageFileReader()
    r.SetFileName(path)
    if load_private:
        r.LoadPrivateTagsOn()
    try:
        r.ReadImageInformation()
    except Exception:
        return None
    if r.HasMetaDataKey(tag):
        return r.GetMetaData(tag)
    return None


def _parse_dicom_floats(s: str | None) -> tuple[float, ...] | None:
    if not s:
        return None
    parts = s.replace(",", "\\").split("\\")
    try:
        return tuple(float(p) for p in parts if p != "")
    except ValueError:
        return None


def read_dicom_series(
    folder: str,
    series_uid: str | None = None,
    debug: bool = False,
) -> tuple[sitk.Image, list[str], str]:
    """
    Read a DICOM series (3D volume) from a folder.

    If series_uid is None, it picks the series with the most slices.
    Returns (image, file_list, chosen_series_uid).
    """
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Not a folder: {folder}")

    series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(folder)
    if not series_ids:
        raise RuntimeError(f"No DICOM series found in: {folder}")

    if series_uid is None:
        # pick the series with the most files (often the actual volume, but not guaranteed)
        best_id = None
        best_n = -1
        for sid in series_ids:
            files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(folder, sid)
            if len(files) > best_n:
                best_n = len(files)
                best_id = sid
        series_uid = best_id

    # IMPORTANT: GetGDCMSeriesFileNames returns a geometry-sorted list
    file_list = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(folder, series_uid)
    if not file_list:
        raise RuntimeError(f"Series UID {series_uid} has no files in: {folder}")

    if debug:
        # Use first slice’s IOP to compute a slice normal; then project IPP onto it to verify order.
        iop_str = _get_dicom_tag_as_str(file_list[0], "0020|0037")
        iop = _parse_dicom_floats(iop_str)  # 6 floats: row(3), col(3)
        normal = None
        if iop and len(iop) == 6:
            row = iop[0:3]
            col = iop[3:6]
            # cross product row x col
            normal = (
                row[1] * col[2] - row[2] * col[1],
                row[2] * col[0] - row[0] * col[2],
                row[0] * col[1] - row[1] * col[0],
            )

        print(f"[DEBUG] Chosen SeriesInstanceUID: {series_uid}")
        print(f"[DEBUG] Number of files: {len(file_list)}")
        if iop:
            print(f"[DEBUG] IOP (0020,0037) from first slice: {iop}")
        if normal:
            print(f"[DEBUG] Slice normal (row x col): {normal}")

        print("[DEBUG] Sorted slices (in returned order):")
        for i, fp in enumerate(file_list):
            inst_str = _get_dicom_tag_as_str(fp, "0020|0013")  # InstanceNumber
            ipp_str = _get_dicom_tag_as_str(fp, "0020|0032")   # ImagePositionPatient
            ipp = _parse_dicom_floats(ipp_str)

            s_coord = None
            if normal and ipp and len(ipp) == 3:
                s_coord = ipp[0] * normal[0] + ipp[1] * normal[1] + ipp[2] * normal[2]

            base = os.path.basename(fp)
            if ipp and len(ipp) == 3:
                if s_coord is None:
                    print(f"  {i:04d}  {base}  Instance={inst_str}  IPP={ipp}")
                else:
                    print(f"  {i:04d}  {base}  Instance={inst_str}  IPP={ipp}  s={s_coord:.6f}")
            else:
                print(f"  {i:04d}  {base}  Instance={inst_str}  IPP=<missing/unparseable>")

    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(file_list)
    
    # Enable metadata loading
    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()
    
    # CRITICAL: This allows for slight variations in slice spacing 
    # which usually causes the (1,1,1) and (0,0,0) fallback.
    reader.SetSpacingPrecision(1e-5) 

    try:
        img3d = reader.Execute()
        
        # VALIDATION: Check if sitk actually gave us default values
        if img3d.GetOrigin() == (0.0, 0.0, 0.0) and img3d.GetSpacing() == (1.0, 1.0, 1.0):
            if debug:
                print("[WARNING] SITK reverted to default geometry. Attempting manual fix...")
            
            # Manual Fix: Extract spacing/origin from the first slice manually
            first_slice = sitk.ReadImage(file_list[0])
            img3d.SetOrigin(first_slice.GetOrigin())
            img3d.SetDirection(first_slice.GetDirection())
            
            # Calculate Z-spacing from the IPP of the first two slices
            # (Note: This assumes the rest are consistent enough for your needs)
            # Better yet: use the 'SetSpacingPrecision' approach above first.
            
    except Exception as e:
        print(f"Failed to read: {e}")
        raise

    # return img3d, file_list, series_uid

    # reader = sitk.ImageSeriesReader()
    # reader.SetFileNames(file_list)

    # # optional but often helpful for DICOM:
    # reader.MetaDataDictionaryArrayUpdateOn()
    # reader.LoadPrivateTagsOn()

    # img3d = reader.Execute()

    if debug:
        print("[DEBUG] Output SimpleITK image geometry:")
        print("  size     :", img3d.GetSize())
        print("  spacing  :", img3d.GetSpacing())
        print("  origin   :", img3d.GetOrigin())
        print("  direction:", img3d.GetDirection())
        # For most series, origin should match IPP of file_list[0] (up to tiny rounding)
        ipp0 = _parse_dicom_floats(_get_dicom_tag_as_str(file_list[0], "0020|0032"))
        print("  IPP(first slice):", ipp0)

    return img3d, file_list, series_uid


In [5]:
mr_img, mr_files, mr_uid = read_dicom_series(folder_mr, debug=True)
print("======================================== MR ========================================")
print(path_dict["MR"])
print("MR series UID:", mr_uid)
print("MR size          :", mr_img.GetSize())
print("MR spacing       :", mr_img.GetSpacing())
print("MR origin        :", mr_img.GetOrigin())
print("MR direction     :", mr_img.GetDirection())

[DEBUG] Chosen SeriesInstanceUID: 2.16.840.1.114362.1.12046989.25631758973.631603556.105.3919
[DEBUG] Number of files: 22
[DEBUG] IOP (0020,0037) from first slice: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0)
[DEBUG] Slice normal (row x col): (0.0, 0.0, 1.0)
[DEBUG] Sorted slices (in returned order):
  0000  2.16.840.1.114362.1.12046989.25631758973.631603556.162.3933.dcm  Instance=1   IPP=(0.0, 0.0, -124.50574493408)  s=-124.505745
  0001  2.16.840.1.114362.1.12046989.25631758973.631603556.160.3932.dcm  Instance=2   IPP=(0.0, 0.0, -118.57689666748)  s=-118.576897
  0002  2.16.840.1.114362.1.12046989.25631758973.631603556.159.3931.dcm  Instance=3   IPP=(0.0, 0.0, -112.64805603027)  s=-112.648056
  0003  2.16.840.1.114362.1.12046989.25631758973.631603556.153.3929.dcm  Instance=4   IPP=(0.0, 0.0, -106.71920776367)  s=-106.719208
  0004  2.16.840.1.114362.1.12046989.25631758973.631603556.151.3928.dcm  Instance=5   IPP=(0.0, 0.0, -100.79036712646)  s=-100.790367
  0005  2.16.840.1.114362.1.12046989.25631

AttributeError: 'ImageSeriesReader' object has no attribute 'SetSpacingPrecision'

In [ ]:
print("Physical (0,0,0)   :", mr_img.TransformIndexToPhysicalPoint((0,0,0)))
print("Physical (0,0,21)  :", mr_img.TransformIndexToPhysicalPoint((0,0,21)))
